# Aula 2, Memória de longo prazo

Notebook executável que acompanha a aula [02-memoria-de-longo-prazo.md](../../lessons/modulo-13-student-modeling/02-memoria-de-longo-prazo.md).

## O que vamos fazer aqui

Um acompanhamento de longo prazo só existe se o sistema lembra do aluno entre sessões. Vamos dar
persistência ao perfil, salvando e carregando em um arquivo JSON, e mostrar a continuidade entre
duas sessões. Python puro.

## Perfil persistente: salvar e carregar

A persistência grava o perfil em disco e o recupera de forma fiel, para a próxima sessão continuar
de onde a anterior parou.

In [ ]:
import json
import tempfile
import os


class PerfilPersistente:
    """Perfil do aluno que pode ser salvo e carregado de um arquivo JSON."""

    def __init__(self, nome, nivel="iniciante", dominio=None):
        self.nome = nome
        self.nivel = nivel
        self.dominio = dominio or {}

    def registrar_resposta(self, tema, correto, passo=0.2):
        atual = self.dominio.get(tema, 0.3)
        atual = min(1.0, atual + passo) if correto else max(0.0, atual - passo)
        self.dominio[tema] = round(atual, 2)

    def salvar(self, caminho):
        with open(caminho, "w", encoding="utf-8") as f:
            json.dump({"nome": self.nome, "nivel": self.nivel, "dominio": self.dominio}, f)

    @classmethod
    def carregar(cls, caminho):
        with open(caminho, encoding="utf-8") as f:
            d = json.load(f)
        return cls(d["nome"], d["nivel"], d["dominio"])


arquivo = os.path.join(tempfile.gettempdir(), "perfil_ana.json")

# Sessão 1: estuda e salva.
ana = PerfilPersistente("Ana")
ana.registrar_resposta("derivada", True)
ana.registrar_resposta("derivada", True)
ana.salvar(arquivo)
print("Sessão 1, domínio salvo:", ana.dominio)

# Sessão 2: carrega e continua.
ana2 = PerfilPersistente.carregar(arquivo)
print("Sessão 2, domínio carregado:", ana2.dominio)
ana2.registrar_resposta("derivada", True)
print("Sessão 2, após nova resposta:", ana2.dominio)

A primeira sessão salva o perfil; a segunda o carrega e a Ana não é mais uma desconhecida,
o sistema lembra que ela domina bem a derivada, e a sessão continua. Essa continuidade entre sessões
é o que permite o acompanhamento de longo prazo. Para o projeto, garanta que o perfil salvo e
carregado seja idêntico.